In [20]:
!python --version
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Python 3.11.3CUDA Available: True

GPU Name: NVIDIA GeForce GTX 1050


In [ ]:
!pip install pytorch_tabular

In [4]:
import pandas as pd
import torch

from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

In [6]:
df = pd.read_csv('jan_data.csv', sep='\t', low_memory=False)

In [15]:
df.head()

,Duration,Service,Source_bytes,Destination_bytes,Count,Same_srv_rate,Serror_rate,Srv_serror_rate,Dst_host_count,Dst_host_srv_count,Dst_host_same_src_port_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Flag,IDS_detection,Malware_detection,Ashula_detection,Label,Source_IP_Address,Source_Port_Number,Destination_IP_Address,Destination_Port_Number,Protocol,IDS_detection_alert_count,Malware_detection_alert_count,Ashula_detection_alert_count
0,1.501374,other,0,0,0,0.0,0.0,0.50,35,35,0.00,0.89,0.89,S0,0,0,0,-1,fd75:41fb:cf76:fca4:019c:41cf:3af2:35c6,7681,fd75:41fb:cf76:39ef:7d8b:279c:615c:0d4d,445,tcp,0,0,0
1,2.589524,other,1621,324,1,1.0,0.0,0.67,1,43,1.00,0.00,0.00,SF,0,0,0,1,fd75:41fb:cf76:36db:202e:034f:077c:556c,37273,fd75:41fb:cf76:dc4c:7d2c:2705:07b2:0f45,25,tcp,0,0,0
2,0.000000,other,0,0,0,0.0,0.0,0.50,23,23,0.04,0.87,0.87,S0,0,0,0,-1,fd75:41fb:cf76:7cb9:4335:184e:535b:1e5a,2158,fd75:41fb:cf76:7f43:7d86:2789:6146:0399,445,tcp,0,0,0
3,10.855827,other,0,90,1,1.0,0.0,0.67,0,44,0.00,0.00,0.00,RSTO,0,0,0,1,fd75:41fb:cf76:532a:0a93:ff00:07cd:2dbc,54567,fd75:41fb:cf76:dc4c:7d2c:2705:07b2:0f45,25,tcp,0,0,0
4,72.309494,other,100,0,0,0.0,0.0,0.50,0,0,0.00,0.00,0.00,S0,0,0,0,-1,fd75:41fb:cf76:57c1:03dd:19c0:2010:7044,46992,fd75:41fb:cf76:b432:7d6b:276f:6080:3945,56674,udp,0,0,0


In [8]:
pd.set_option('display.max_columns', 200)

In [9]:
df = df.drop('Start_Time', axis=1)
df = df.drop('Unnamed: 0', axis=1)

In [10]:
df['Ashula_detection'] = df['Ashula_detection'].astype('str')
df['Label'] =df['Label'].astype('str')
df['Source_Port_Number'] = df['Source_Port_Number'].astype('str')
df['Destination_Port_Number'] = df['Destination_Port_Number'].astype('str')
df['Label'] = df['Label'].astype('str')

In [11]:
CAT_FEATURES = ['Service',
                'Flag',
                'IDS_detection',
                'Malware_detection',
                'Ashula_detection',
                'Source_IP_Address',
                'Source_Port_Number',
                'Destination_IP_Address',
                'Destination_Port_Number',
                'Protocol'
               ]
NUM_FEATURES = [
    'Destination_bytes',
    'Duration',
    'Dst_host_serror_rate',
    'Dst_host_srv_serror_rate',
    'Dst_host_srv_count',
    'Source_bytes',
    'Dst_host_count',
    'Serror_rate',
    'Count',
    'IDS_detection_alert_count',
    'Ashula_detection_alert_count',
    'Dst_host_same_src_port_rate',
    'Malware_detection_alert_count',
    'Srv_serror_rate',
    'Same_srv_rate'
]

In [12]:
from sklearn.model_selection import train_test_split

train_features = NUM_FEATURES + CAT_FEATURES
X = df[train_features]
y = df['Label']

In [13]:
X.shape

(3439069, 25)

In [14]:
y.shape

(3439069,)

In [16]:
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=32)

In [17]:
x_train.shape

(2579301, 25)

In [18]:
y_train.shape

(2579301,)

In [19]:
x_test.shape

(859768, 25)

In [27]:
data_config = DataConfig(
    target=["Label"],
    continuous_cols=NUM_FEATURES,  # Numerical features
    categorical_cols=CAT_FEATURES,  # Categorical features
)

trainer_config = TrainerConfig(
    accelerator="gpu",
    devices=-1,
    precision=16,
    auto_lr_find=True,
    batch_size=32,
    max_epochs=1,
)

optimizer_config = OptimizerConfig()

model_config = TabTransformerConfig(
    task="classification",
    input_embed_dim=32,  # Embedding dimension
    num_heads=4,  # Number of attention heads
    num_attn_blocks=6,  # Number of transformer blocks
    learning_rate=1e-3,
)


tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

trainer = x_train.join(y_train)
tabular_model.fit(train=trainer)

Epoch 0/0  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64483/64483 1:21:39 • 0:00:00 13.67it/s v_num: 2.000 train_loss: nan

`Trainer.fit` stopped: `max_epochs=1` reached.


2025-04-01 01:20:00,130 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-01 01:20:00,145 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

In [28]:
tabular_model.save_model("TabularIDSModel")

In [32]:
!pip install torchinfo

In [41]:
print(tabular_model.ret_summary())

    | Name                                                                | Type                    | Params | Mode 
--------------------------------------------------------------------------------------------------------------------------
0   | _backbone                                                           | TabTransformerBackbone  | 173 K  | train
1   | _backbone.transformer_blocks                                        | Sequential              | 172 K  | train
2   | _backbone.transformer_blocks.mha_block_0                            | TransformerEncoderBlock | 28.8 K | train
3   | _backbone.transformer_blocks.mha_block_0.mha                        | MultiHeadedAttention    | 16.4 K | train
4   | _backbone.transformer_blocks.mha_block_0.mha.to_qkv                 | Linear                  | 12.3 K | train
5   | _backbone.transformer_blocks.mha_block_0.mha.to_out                 | Linear                  | 4.1 K  | train
6   | _backbone.transformer_blocks.mha_block_0.mha.dropout

In [43]:
tabular_model.config.loss

'CrossEntropyLoss'

In [65]:
import os

os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"